In [1]:
import kagglehub, os, glob, hashlib
IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
CKPT = kagglehub.dataset_download("ashwanisingh997/run1-checkpoints-1229692")

src = {f: open(f"{CODE}/{f}").read() for f in ["memorization.py", "checkpointing.py", "diffusion.py"]}
assert "def verify_equivalence" in src["memorization.py"], "code is not v5"
assert 'format={"jpg"' in src["checkpointing.py"] and "pipe.vae.enable_slicing()" in src["diffusion.py"]
print("code: v5 (6c8939b) confirmed")

rows = []
for dp, _, files in os.walk(CKPT):
    for f in files:
        full = os.path.join(dp, f); h = hashlib.sha256()
        with open(full, "rb") as fh:
            for ch in iter(lambda: fh.read(1 << 20), b""): h.update(ch)
        rows.append((os.path.relpath(full, CKPT).replace(os.sep, "/"), h.hexdigest()))
rows.sort()
dig = hashlib.sha256("\n".join(f"{p}|{s}" for p, s in rows).encode()).hexdigest()
assert dig == "81280427151b71c7eb1cc79d78415265d6924ec30c27210ee0bd0c708672dbf2", "CHECKPOINTS DIFFER"
print("checkpoints: MATCH")

code: v5 (6c8939b) confirmed
checkpoints: MATCH


In [2]:
!pip uninstall -y -q torchao
import importlib; importlib.invalidate_caches()
import peft.import_utils as piu
if hasattr(piu.is_torchao_available, "cache_clear"): piu.is_torchao_available.cache_clear()
assert piu.is_torchao_available() is False
print("CELL 2 OK")

CELL 2 OK


In [3]:
import sys, shutil, gc, torch, yaml
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True); os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    shutil.copy(f"{CODE}/{f}", f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}")
sys.path.insert(0, WORK); os.chdir(WORK)
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11")
cfg = yaml.safe_load(open("config/config.yaml"))
assert cfg["synthesis"]["curve_per_class"] == 500
from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="6c8939b")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud


In [4]:
from src.data import AUG_CLASSES
from src.gan import Generator
from src.diffusion import BASE, build_pipeline, generate_diffusion, generate_gan_images
from src.checkpointing import verify_synthetic_set
from diffusers import UNet2DConditionModel
from peft import PeftModel

PART  = "a"    # "a": 1.0h and 2.0h.  "b": 3.0h and 4.0h.
SMOKE = False   # True: 2 images per class as a test. False: the real 500 per class.
LABELS = {"a": ["1.0h", "2.0h"], "b": ["3.0h", "4.0h"]}[PART]
N   = 2 if SMOKE else cfg["synthesis"]["curve_per_class"]
TAG = "_smoke" if SMOKE else ""
OUT = f"/kaggle/working/curve_{PART}{TAG}"
CK, LOG = f"{CKPT}/checkpoints", f"{root}/logs"
s = cfg["diffusion"]["sampler"]
print(f"part {PART}: checkpoints {LABELS}, {N} per class -> {OUT}")

for lab in LABELS:
    G = Generator(latent_dim=cfg["gan"]["latent_dim"])
    G.load_state_dict(torch.load(f"{CK}/gan_{lab}.pt", map_location="cpu", weights_only=False)["models"]["G"])
    generate_gan_images(G, AUG_CLASSES, N, f"{OUT}/curve_{lab}/gan", latent_dim=cfg["gan"]["latent_dim"],
                        base_seed=42, manifest_path=f"{LOG}/sampling_curve_gan_{lab}{TAG}.json")
    verify_synthetic_set(f"{OUT}/curve_{lab}/gan", AUG_CLASSES, N)
    del G; gc.collect(); torch.cuda.empty_cache()

for lab in LABELS:
    unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16)
    unet = PeftModel.from_pretrained(unet, f"{CK}/lora_{lab}")
    n_lora = sum(p.numel() for n, p in unet.named_parameters() if "lora_" in n)
    assert n_lora > 1e6, f"adapter {lab} did not attach"
    print(f"[check] lora_{lab}: {n_lora/1e6:.2f}M LoRA parameters")
    pipe = build_pipeline(unet.merge_and_unload().to(torch.float16))
    generate_diffusion(pipe, AUG_CLASSES, N, f"{OUT}/curve_{lab}/lora", base_seed=42,
                       steps=s["num_inference_steps"], guidance=s["guidance_scale"],
                       size=cfg["data"]["image_size_diffusion"], target_size=cfg["data"]["image_size_classifier"],
                       manifest_path=f"{LOG}/sampling_curve_lora_{lab}{TAG}.json", session_limit_hours=10.5)
    verify_synthetic_set(f"{OUT}/curve_{lab}/lora", AUG_CLASSES, N)
    del pipe, unet; gc.collect(); torch.cuda.empty_cache()
print("CURVE SAMPLING COMPLETE")

part a: checkpoints ['1.0h', '2.0h'], 500 per class -> /kaggle/working/curve_a
[sample] akiec: 50 made, 0 skipped, 0.02s/image, ~0.02h left
[sample] akiec: 100 made, 0 skipped, 0.02s/image, ~0.01h left
[sample] akiec: 150 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 200 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 250 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 300 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 350 made, 0 skipped, 0.01s/image, ~0.01h left
[sample] akiec: 400 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] akiec: 450 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] akiec: 500 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] bcc: 550 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] bcc: 600 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] bcc: 650 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] bcc: 700 made, 0 skipped, 0.01s/image, ~0.00h left
[sample] bcc: 750 made, 0 skipped, 0.01s/image, ~0.00h left
[s

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

[check] lora_1.0h: 1.59M LoRA parameters


... (9 outputs left out)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[sample] akiec: 50 made, 0 skipped, 5.49s/image, ~3.74h left
[sample] akiec: 100 made, 0 skipped, 5.49s/image, ~3.66h left
[sample] akiec: 150 made, 0 skipped, 5.48s/image, ~3.58h left
[sample] akiec: 200 made, 0 skipped, 5.49s/image, ~3.51h left
[sample] akiec: 250 made, 0 skipped, 5.48s/image, ~3.43h left
[sample] akiec: 300 made, 0 skipped, 5.49s/image, ~3.35h left
[sample] akiec: 350 made, 0 skipped, 5.48s/image, ~3.27h left
[sample] akiec: 400 made, 0 skipped, 5.48s/image, ~3.20h left
[sample] akiec: 450 made, 0 skipped, 5.48s/image, ~3.12h left
[sample] akiec: 500 made, 0 skipped, 5.48s/image, ~3.05h left
[sample] bcc: 550 made, 0 skipped, 5.48s/image, ~2.97h left
[sample] bcc: 600 made, 0 skipped, 5.48s/image, ~2.89h left
[sample] bcc: 650 made, 0 skipped, 5.48s/image, ~2.82h left
[sample] bcc: 700 made, 0 skipped, 5.48s/image, ~2.74h left
[sample] bcc: 750 made, 0 skipped, 5.48s/image, ~2.66h left
[sample] bcc: 800 made, 0 skipped, 5.48s/image, ~2.59h left
[sample] bcc: 850 mad

In [5]:
shutil.make_archive(f"/kaggle/working/run1_curve_{PART}{TAG}", "zip", "/kaggle/working", f"curve_{PART}{TAG}")
shutil.make_archive(f"/kaggle/working/run1_curve_{PART}{TAG}_logs", "zip", "/kaggle/working/results")
print("saved zips")

saved zips
